# Investigator Agent + Adaptive Triage - runnable notebook

**Separate, optional layer** on top of the base Crypto-AML pipeline. The base
project (notebook, `pipeline_extended.py`, `streamlit_app.py`, ...) runs on its
own and is never modified. This notebook is how you drive the AI layer.

It covers both components from `docs/SPEC_investigator_and_adaptive_triage.md`:

- **A - Investigator agent:** a ReAct loop over 7 tools that writes a cited
  markdown dossier for one wallet, under strict budgets.
- **B - Adaptive triage:** analyst verdicts -> online LinUCB reranker,
  warm-started by an offline XGBoost pairwise LTR pass.

> Every output is a **research lead, not a finding of guilt.**

### Prerequisites
- Base project run at least once (so `suspicious_wallets_master.csv` exists).
- For a **real** agent run: `BQ_PROJECT` set (BigQuery) + `ANTHROPIC_API_KEY`.
- For an **offline demo**: set `INVESTIGATOR_LLM_PROVIDER=mock` (no key, no cost).

## 0. Setup

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

# Run this notebook from inside investigator/ OR from the project root - either
# way, make the project root importable so `import investigator` resolves.
from pathlib import Path
here = Path.cwd()
root = here if (here / 'investigator').exists() else here.parent
sys.path.insert(0, str(root))

# ---- choose your provider here -------------------------------------------
# 'mock'      -> fully offline, no key, deterministic (great for a first run)
# 'anthropic' -> real Claude agent (needs ANTHROPIC_API_KEY)
os.environ.setdefault('INVESTIGATOR_LLM_PROVIDER', 'mock')
# --------------------------------------------------------------------------

from investigator import config as C
C.summary()

## 1. Component A - investigate a single wallet

Pick a seed wallet (default: the top-actionability row from the master), run the
agent, and render the dossier it writes. With `provider='mock'` this runs fully
offline; the graph tool will report `bigquery_unavailable` and the agent simply
proceeds with the other tools.

In [ ]:
import pandas as pd
from investigator.agent import Investigator
from investigator.llm_client import LLMClient

master = pd.read_csv(C.MASTER_CSV)
seed = master.sort_values('actionability', ascending=False).iloc[0]
address, chain = str(seed['wallet']), str(seed['chain'])
print('seed:', address, '|', chain)

inv = Investigator(llm=LLMClient(), master=master, verbose=True)
result = inv.investigate(address, chain)
print('\nstatus:', 'PARTIAL' if result.partial else 'complete',
      '| stop:', result.stop_reason, '| sections:', result.n_sections,
      '| valid:', result.valid)
print('BQ $%.4f | tokens %d | %.1fs' % (result.bq_cost_usd, result.token_cost['total'], result.duration_s))

In [ ]:
from IPython.display import Markdown
Markdown(Path(result.dossier_md_path).read_text(encoding='utf-8'))

### Inspect the reasoning trace
Every `[TOOL:n]` in the dossier maps to one of these calls.

In [ ]:
import json
trace = json.loads(Path(result.trace_json_path).read_text(encoding='utf-8'))
for i, c in enumerate(trace['trace'], 1):
    status = 'ERR ' + c['error'] if c.get('error') else 'ok'
    print(f"[TOOL:{i}] {c['tool']:<16} {status}")

## 2. Component B - adaptive triage

Build the fixed-length feature vectors, feed analyst verdicts into the online
LinUCB, and watch the queue re-rank. Below the cold-start floor
(`MIN_VERDICTS_TO_ACTIVATE`) the reranker keeps the static `actionability`
order, so flipping the flag is safe.

In [ ]:
from investigator.adaptive_triage import FeatureBuilder, LinUCB, Reranker, FEATURE_DIM
from investigator import verdicts_io

fb = FeatureBuilder(FeatureBuilder.cluster_vocab_from_master(master))
X = fb.build_frame(master.head(5))
print('context vectors:', X.shape, '(fixed dim =', FEATURE_DIM, ')')

In [ ]:
# --- simulate a few verdicts (in real use these come from the Streamlit app) --
# label top-actionability as positive leads, bottom as legitimate services
def feats(r):
    return {k: (v if isinstance(v,(int,float,bool,str)) or v is None else str(v)) for k,v in r.items()}

lin = LinUCB()
ranked = master.sort_values('actionability', ascending=False)
for _, r in ranked.head(20).iterrows():
    lin.update(fb.build(feats(r)), verdicts_io.reward_for('real_informal_exchanger'))
for _, r in ranked.tail(20).iterrows():
    lin.update(fb.build(feats(r)), verdicts_io.reward_for('legitimate_service'))
print('LinUCB updates:', lin.n_updates)

rk = Reranker(linucb=lin, feature_builder=fb, flag_enabled=True, min_verdicts_to_activate=30)
print('reranker active:', rk.active())
out = rk.rerank(master[master['chain'] == 'ethereum'])
out[['wallet','chain','risk_score','actionability','bandit_score']].head(8)

### Offline LTR + warm-start (XGBoost pairwise -> LinUCB prior)

Collect real verdicts through the app, then this pass trains the ranker and
seeds LinUCB with a linear prior so it does not start cold.

In [ ]:
from investigator import train_ltr
# For a real pass, verdicts come from verdicts.jsonl (via the Streamlit app or
# backfill_verdicts.py). This demo just shows the ridge warm-start weights.
summary = train_ltr.train(master=master, apply_warm_start=False, verbose=True)
summary.get('top_features')

## 3. Batch mode (optional)

Re-rank each chain, take the top-K, and write a dossier for each into
`investigator/outputs/dossiers/<chain>/`, plus an `index.html`. Keep `top_k`
small on the first run. With `provider='mock'` this is free and offline.

In [ ]:
from investigator import run_nightly
idx = run_nightly.run(top_k=2, chains=['ethereum'], provider='mock', verbose=True)
print('\nwrote', idx['n_dossiers'], 'dossiers | index:', C.DOSSIER_INDEX_PATH)

## 4. Interactive dashboard

For the verdict widget + on-demand dossiers in a UI, run the **separate**
Streamlit app from a terminal (not from this notebook):

```bash
streamlit run investigator/app.py
```

The base dashboard (`streamlit run streamlit_app.py`) is unaffected and still
works on its own.

---

### Ethics
Dossiers are investigative **research leads**, never findings of guilt. Every
section carries that notice, every claim cites a tool observation, and the full
reasoning trace ships with each dossier. Identity attribution to a person stays
with authorised enforcement (KYC subpoena to the exchange).